# Milvus? ??? ??? ??? ?? ? ??

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# VectorDB ? Milvus
Milvus ? VectorDB? ??? ??? ??? ???? ???? ??? ??? ? ??? ??? ????. ?? ???? ??? ??? BERT ??? ?? ??? ???? ??? ??? ?? ?? ????? ??????.


### ???? ????
"wikihow.csv" ????? ???? ?? ??? ?? ?????. ???? ??: `https://ibm.box.com/s/8nvanf974t35d89cmibk75e3gc6d1pbo` [TOBECHANGED]

In [ ]:

WH_PATH = "wikihow.csv"
WH_PATH

### ??? ?? ? ??? ???
?? ??? ????? ??? ?? ??? ????:
- titles ???? sectionLabel ???? ??? ??
- ??????? ?? ?? ??? ??? ?? ??
- ?? ??? ??? ?? ??

In [ ]:
import pandas
doc = pandas.read_csv(WH_PATH)

In [ ]:
doc_indexed = doc.set_index(['title', 'headline']).sort_index()

In [ ]:
doc_indexed.tail()

### Milvus DB connection 
Python?? MilvusDB ???? ???? ?? ?????? pymilvus? ??? ?????. ?? LangChain? Milvus ????? ???? ???? ?? ????? ??? ?? ????. 

In [ ]:
# Milvus requires a connection for all operations. Remember to disconnect at the end.

from pymilvus import connections
connections.connect(
  alias="default",
  host="127.0.0.1", # YOUR IP
  port="19530"      # YOUR PORT
)

### Milvus store? DB schema ??
?? collection ??? ??? ?? schema? ??? ?? SchemaNotReady exception? ??? ? ???, collection? schema? pair? ????? ???.
?? ??? ??? ?? ??? ??? ?? ??? ??? ?????. ???? ??? ??? ??? ???? ??? ??? ?? ???? DB??? ??? ?? ?? ???, DB? ??? record?? ??? ??? ?? ?? ??? ?? ??? ???? ???(65535)?? ??? ???? ?? ????.


?? ??? ??? ??? namespace? ????? ???.
- primary key? ??? "pk"? ????? ???.
- ??? vector? "vector"? ????? ???.
- text entry ? ??? "text"? ????? ???.

In [ ]:
from pymilvus import CollectionSchema, FieldSchema, DataType, Collection, utility

In [ ]:
# Milvus also supports schemaless operations if `enable_dynamic_fields=True`.

MAX_TITLE = 512
MAX_TEXT = 1024
MAX_VEC = 384

NAME = "WikiHow"

if NAME in utility.list_collections():
    whcollection = Collection(NAME)
    whcollection.drop()

whschema = CollectionSchema(
    fields=[
        FieldSchema(name="pk", dtype=DataType.INT64, is_primary=True, auto_id=True),
        FieldSchema(name="title", dtype=DataType.VARCHAR, max_length=65535, default_value=""),
        FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535, default_value=""),
        FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=384)
    ],
    enable_dynamic_fields=False,
    description="WikiHow collection"
)
whcollection = Collection(
    name=NAME,
    schema=whschema,
    consistency_level="Session" # Make sure we read our own writes, otherwise allowed to be a bit out of date.
)

### MilvusDB ??? Batch ?? ??
?? BERT ??? ???? Milvus store? ??? ??? ?? ???? ??? ?????.

? ???? ?? ?? ??? ??? ???????. ???? ??? ??? ?? ??? Batch ?? ????? ?? ??? ??? ?? ? ????.



??? ??? GPU?? ???? ??, ?? ??? ????? ??? ??? ????? ???? ? ??? ???? ???.(?? ??? ?? ??? ??? ????? ?? CPU,GPU? ??? ??? ?? ??? ????, ?? ?? ??? ? ????).
????? ?? ??? ????? ??, ?? ???? ???? API ?? ??? ?? ?? ????? ??? ? ????.


MilvusDB??? record ?? ?? ??? ?? ??? ?? ????? ?? MilvusDB? ?? ????? ??? ? ????. ?? ??? ??? ? ?? ???? ???? ???? ???? ???? ???? ?? ? ?? ???? ?? ??? ??? ? ?? record ??? ?????.


### ??? ?? ??
HuggingFaceEmbeddings? MiniLM BERT model? ???? ???? ?????.

In [ ]:
import langchain
from langchain.embeddings import HuggingFaceEmbeddings

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

In [ ]:
BATCH_SIZE = 2048

batch = []
def insert_data(data):
    import math

    batch = []

    titles = list(data.keys())

    vecs = embeddings.embed_documents(titles)
    
    entries = [[], [], []]

    for b, title in enumerate(titles):
        text = title + ":\n"
        for cat in data[title]:
            text += cat + ":\n"
            text += "\n".join(data[title][cat])
            
        title_len_diff = len(title.encode('utf-16-le')) - len(title)
        text_len_diff = len(text.encode('utf-16-le')) - len(text)
        entries[0].append(title[:MAX_TITLE - title_len_diff])
        entries[1].append(text[:MAX_TEXT - text_len_diff])
        entries[2].append(vecs[b])

    whcollection.insert(entries)

import collections, tqdm
doc_data = collections.defaultdict(lambda: collections.defaultdict(list))
for i in tqdm.tqdm(range(len(doc_indexed)), total=len(doc_indexed)):
    if (type(doc_indexed.index[i][0]) is not str) or (type(doc_indexed.index[i][1]) is not str):
        continue
    die = False
    for col in ['text', 'overview', 'sectionLabel']:
        if type(doc_indexed.iloc[i][col]) is not str:
            die = True
            break
    if die:
        continue
    section_head = doc_indexed.index[i][0] + " (" + doc_indexed.iloc[i]['overview'].strip() + ")"
    category = doc_indexed.index[i][1]
    step = " ".join(map(lambda x: x.strip(), doc_indexed.iloc[i][['sectionLabel', 'text']]))

    if len(doc_data) % BATCH_SIZE == 1 and len(doc_data) != 1:
        insert_data(doc_data)
        doc_data = collections.defaultdict(lambda: collections.defaultdict(list))
    doc_data[section_head][category].append(step)
    if i == len(doc_indexed) - 1:
        insert_data(doc_data)

In [ ]:
# Milvus will not seal segments that are too small, a flush is necessary to force it.
whcollection.flush()

## Vector ???? ??
??? ????? ?? ??? ?? ?? ? ????. ???? L2 norm? flat indexing? ?????. (????: `IVF_FLAT`).

collection? ???? ?? Milvus? ???? ?? collection? ???? ??? connection?? ?? collection? ?? ??? ??? ??? ?? ???.
?? langchain milvus store interface? ??? ??? ??? ??? ???? ???.



In [ ]:
whcollection.create_index(field_name="vector", index_params={"metric_type": "L2", "index_type": "IVF_FLAT", "nlist": "1024"})

In [ ]:
whcollection.load()
# To actually use the data, we would have to do a `whcollection.load()` before any queries.
# Once done with queries, we should then use `whcollection.release()` to stop using resources

## Disconnect
DB? ?? ? ??? connection? ??? ?? ???? ????.

In [ ]:
whcollection.release()
connections.disconnect("default")